In [1]:
from pyspark.sql.types import *
import pyspark.sql.functions as F
# This is a helper cell to create a spark session if it doesn't exist. 
# On Databricks website: spark already exists → try succeeds → DatabricksSession is never imported → no conflict
# In VS Code: spark doesn't exist → falls into except → creates the session
try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()

###

# I. With StringType() and ArrayType()

### I.1. StringType() and using F.from_json()

* In this example: stringtype will be written as a dictionary

In [2]:
schema= StructType([
    StructField("CustomerID", IntegerType(), False),
    StructField("json_data", StringType(), True)
])

In [3]:
# get data as a list of rows
data= [
    Row(CustomerID= 1,
        json_data= {"name": "ALice", "age": 25}),
    
    Row(CustomerID= 2,
        json_data= {"name": "Bob", "age": 30})
]

In [4]:
df= spark.createDataFrame(data, schema)

In [5]:
df.printSchema()

root
 |-- CustomerID: integer (nullable = false)
 |-- json_data: string (nullable = true)



In [6]:
df.show(truncate=False)

+----------+----------------------------+
|CustomerID|json_data                   |
+----------+----------------------------+
|1         |{'name': 'ALice', 'age': 25}|
|2         |{'name': 'Bob', 'age': 30}  |
+----------+----------------------------+



* Now we want to convert the column json_data into a nested structure StrucType() by using from_json(column, schema) function

In [7]:
schema_column= StructType([
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True)
])

In [8]:
df_new= df.select(
    "CustomerID",
    F.from_json("json_data", schema_column).alias("CustomerInfo")
)

In [9]:
df_new.show(truncate= False)

+----------+------------+
|CustomerID|CustomerInfo|
+----------+------------+
|1         |{ALice, 25} |
|2         |{Bob, 30}   |
+----------+------------+



In [10]:
df_new.printSchema()

root
 |-- CustomerID: integer (nullable = false)
 |-- CustomerInfo: struct (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- age: integer (nullable = true)



### I.2. StructType() and using F.json_tuple()
* Exactly the same like df in I.1 but the column is not a string but a struct type.


In [11]:

df.show(truncate=False)

+----------+----------------------------+
|CustomerID|json_data                   |
+----------+----------------------------+
|1         |{'name': 'ALice', 'age': 25}|
|2         |{'name': 'Bob', 'age': 30}  |
+----------+----------------------------+



In [12]:
df_new= df.select(
    "CustomerID",
    F.json_tuple("json_data", "name", "age").alias("name", "age")
)

In [14]:
df_new

,CustomerID,name,age
0,1,ALice,25
1,2,Bob,30
